In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from scipy.stats import spearmanr, rankdata
from joblib import Parallel, delayed
import warnings
import gc
import argparse
from tqdm.auto import tqdm
import glob
import matplotlib.pyplot as plt
import networkx as nx
import re
import sys

In [15]:
#Path to TwINFER code repository
path_to_code_repo = "/home/gzu5140/Keerthana_b1042/grnInference/code/TwINFER/"
path_to_simulation_data = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/three_gene_sim_all_variants/"
path_to_save_plot_data = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4_19022026/"
os.makedirs(path_to_save_plot_data, exist_ok = True)

In [3]:
#Common path to data files
path_to_input_data = f"{path_to_code_repo}/simulation_example_input_data/"
# Note: This configuration need not match the exact simulation parameters.
# However, ensure the following requirements are met:
# 1. The connectivity matrix has the same number of genes as the simulations being analyzed
#    (multiple base_configs can be created if needed for analyzing networks with different number of genes)
# 2. The twin simulation duration matches the time specified in base_config
# 3. The number of twin pairs in the simulation equals n_cell (the number of parent cells)

base_config = {
    'n_cells': 6000, #Number of cells before division (number of twin pairs)
    'simulation_time_before_division': 6000, #The time used to run the initial cells before division. User must set this time to ensure the population reaches steady state [hours]
    'twin_simulation_time_after_division': 48, #The time twin cells are simulated after division and measurements are stored in the output[hours]
    'twin_measurement_resolution': 1, #The time between each measurement of twin cells [hours]. For example, if twin_sampling_duration is 12 and twin_measurement_resolution is 1, the final dataframe will contain hourly measurements for 12 hours (0 is birth).
    "path_to_connectivity_matrix": "/home/gzu5140/TwINFER-main 3/Matrix/Fan_out_example12.txt", #path to the connectivity matrix specifying the GRN to simulate
    "param_csv": f"{path_to_input_data}/median_parameter.csv", #Path to the parameters for all genes and interaction terms
    "rows_to_use": [[0,0,0]], #Rows in the parameter's csv file for each gene - the length should be equal to number of genes in the system. Example - [0,0] will mean use row 0 parameters for both gene 1 and 2
    "output_folder": f"{path_to_simulation_data}", #Path to folder to store simulation 
    "log_file": f"{path_to_code_repo}/example_simulation_output/example_log.jsonl", #Path to the log file
    "type": "Fan_out",  # Name of the network used -- will be in the filename
    "number_of_parallel_parameters": 1, #Number of parameters to be run in parallel
    "number_of_cores_per_parameter": 10, #Number of cores to be used per parameter (number_of_parallel_parameters * number_of_cores_per_parameter = number of cores in your computer)
}

#Default settings for when the two samples are measured
t1 = 1
t2 = 20

## Helper functions - imports and definitions

### Imports for helper functions

In [4]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    read_input_matrix,
    split_and_merge_simulations
)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

### Helper and organising functions defined in this notebook

In [5]:
def get_many_random_pair_corr(
    path_to_simulation_file,
    base_config,
    t1,
    t2,
    threshold_gene_gene_corr=0.04,
    check_for_steady_state=True,
    plot_correlation_matrices_as_heatmap=True,
    have_any_output=True,
    random_seed=42,
):
    """
    Compute large-scale random-pair correlations across clones and time points
    to generate a null distribution for statistical inference.
    Parameters
    ----------
    path_to_simulation_file : str or list[str]
        Simulation input. Accepted formats:
        - str : a single CSV simulation file.
        - list/tuple with 1 file   : treated as a single CSV (loaded directly).
        - list/tuple with >=2 files: passed to `split_and_merge_simulations()`
                                    to merge clone sets across simulations.
        All files must contain the columns:
        ['clone_id', 'time_step', 'replicate', ...].
    base_config : dict
        Configuration dictionary with required keys:
        - "path_to_connectivity_matrix" : str
        - "param_csv" : str
        - "n_cells" : int
        - "twin_simulation_time_after_division" : int/float
        - "twin_measurement_resolution" : int/float
    t1, t2 : int or float
        Measurement time points.
    threshold_gene_gene_corr : float, optional
        Unused in this function; included for interface consistency.
    check_for_steady_state : bool, optional
        Unused here.
    plot_correlation_matrices_as_heatmap : bool, optional
        Unused here.
    have_any_output : bool, optional
        Unused here.
    random_seed : int, optional
        Seed for reproducible clone shuffling.
    Returns
    -------
    random_stats : dict
        Statistical summary of the shuffled null correlation distribution for
        the pair ("gene_1", "gene_2"):
        - 'all_values'     : flatten of all shuffle correlations
        - 'mean_per_pair'  : per-shuffle mean
        - 'std_per_pair'   : per-shuffle std
        - 'percentile_95'  : 95th percentile of |correlation|
        - 'percentile_100' : max of |correlation|
        - 'global_mean'    : mean over all values
        - 'global_std'     : std over all values
    """
    # --------------------------------------------------
    # Load simulation file(s)
    # --------------------------------------------------
    try:
        if isinstance(path_to_simulation_file, str):
            # Single file
            simulation = pd.read_csv(path_to_simulation_file)
        elif isinstance(path_to_simulation_file, (list, tuple)):
            # List of files
            if len(path_to_simulation_file) == 1:
                # Single file inside list → read directly
                simulation = pd.read_csv(path_to_simulation_file[0])
            elif len(path_to_simulation_file) >= 2:
                # Merge mutliple simulation files
                simulation = split_and_merge_simulations(path_to_simulation_file)
            else:
                raise ValueError("List of simulation files is empty.")
        else:
            raise TypeError("path_to_simulation_file must be a string or list/tuple.")
    except Exception as e:
        raise RuntimeError(f"Error reading simulation file(s): {e}")
    # --------------------------------------------------
    # Load connectivity + parameters
    # --------------------------------------------------
    path_to_connectivity_matrix = base_config["path_to_connectivity_matrix"]
    param_df = pd.read_csv(base_config["param_csv"], index_col=0)
    # Read the interaction matrix and number of genes
    n_genes, interaction_matrix = read_input_matrix(path_to_connectivity_matrix)
    gene_list = [f"gene_{i}" for i in np.arange(1, n_genes + 1)]
    # --------------------------------------------------
    # Basic information & sanity checks
    # --------------------------------------------------
    n_clones_simulation = simulation["clone_id"].nunique()
    time_points_base_config = np.arange(
        0,
        base_config["twin_simulation_time_after_division"] +
        base_config["twin_measurement_resolution"],
        base_config["twin_measurement_resolution"],
    )
    # --------------------------------------------------
    # Randomly split clones into 1:1:2 partition
    # --------------------------------------------------
    np.random.seed(random_seed)
    clone_ids_shuffled = np.random.permutation(n_clones_simulation)
    n1 = n2 = n_clones_simulation // 4
    t1_clones = clone_ids_shuffled[:n1]
    t2_clones = clone_ids_shuffled[n1:n1+n2]
    across_t_clones = clone_ids_shuffled[n1+n2:]
    # Data subsets for each partition
    t1_twins = simulation[(simulation["clone_id"].isin(t1_clones)) &
                          (simulation["time_step"] == t1)]
    t2_twins = simulation[(simulation["clone_id"].isin(t2_clones)) &
                          (simulation["time_step"] == t2)]
    across_t_twin1 = simulation[(simulation["clone_id"].isin(across_t_clones)) &
                                (simulation["time_step"] == t1) &
                                (simulation["replicate"] == 1)]
    across_t_twin2 = simulation[(simulation["clone_id"].isin(across_t_clones)) &
                                (simulation["time_step"] == t2) &
                                (simulation["replicate"] == 2)]
    # Reset indices before concatenation
    t1_twins = t1_twins.reset_index(drop=True)
    t2_twins = t2_twins.reset_index(drop=True)
    across_t_twin1 = across_t_twin1.reset_index(drop=True)
    across_t_twin2 = across_t_twin2.reset_index(drop=True)
    # Combine all partitions
    all_t1_t2_measurements = pd.concat(
        [t1_twins, t2_twins, across_t_twin1, across_t_twin2],
        ignore_index=True
    )
    # --------------------------------------------------
    # Generate shuffled random-pair correlations
    # --------------------------------------------------
    scrambled_random_corr = generate_random_shuffle(
        all_t1_t2_measurements,
        gene_list,
        n_shuffles=10000,
        random_state=42,
    )
    # Only use ("gene_1", "gene_2") to match expected structure
    all_correlations_g1_g2 = scrambled_random_corr[("gene_1", "gene_2")]
    all_correlations_g2_g3 = scrambled_random_corr[("gene_2", "gene_3")]
    all_correlations_g1_g3 = scrambled_random_corr[("gene_1", "gene_3")]
    # --------------------------------------------------
    # Compute summary statistics
    # --------------------------------------------------
    random_stats = {
        ("gene_1", "gene_2"):{
        "all_values": all_correlations_g1_g2.flatten(),
        "mean_per_pair": np.mean(all_correlations_g1_g2, axis=0),
        "std_per_pair": np.std(all_correlations_g1_g2, axis=0),
        "percentile_95": np.percentile(np.abs(all_correlations_g1_g2.flatten()), 95),
        "percentile_100": np.percentile(np.abs(all_correlations_g1_g2.flatten()), 100),
        "global_mean": np.mean(all_correlations_g1_g2),
        "global_std": np.std(all_correlations_g1_g2)
        },
        ("gene_2", "gene_3"):{
        "all_values": all_correlations_g2_g3.flatten(),
        "mean_per_pair": np.mean(all_correlations_g2_g3, axis=0),
        "std_per_pair": np.std(all_correlations_g2_g3, axis=0),
        "percentile_95": np.percentile(np.abs(all_correlations_g2_g3.flatten()), 95),
        "percentile_100": np.percentile(np.abs(all_correlations_g2_g3.flatten()), 100),
        "global_mean": np.mean(all_correlations_g2_g3),
        "global_std": np.std(all_correlations_g2_g3)
        },
        ("gene_1", "gene_3"):{
        "all_values": all_correlations_g1_g3.flatten(),
        "mean_per_pair": np.mean(all_correlations_g1_g3, axis=0),
        "std_per_pair": np.std(all_correlations_g1_g3, axis=0),
        "percentile_95": np.percentile(np.abs(all_correlations_g1_g3.flatten()), 95),
        "percentile_100": np.percentile(np.abs(all_correlations_g1_g3.flatten()), 100),
        "global_mean": np.mean(all_correlations_g1_g3),
        "global_std": np.std(all_correlations_g1_g3)
        },
    }
    return random_stats

In [6]:
def process_simulation_through_infer_with_twinfer(path_to_simulation_file, sim_type, rep_id, base_config, t1, t2, gene_names=None):
    """
    Run data from a simulation through TwINFER, extract all correlation matrices, and flatten
    them into gene-pair columns for downstream CSV export.

    Parameters
    ----------
    path_to_simulation_file : str or list
        Path or list of paths to simulation file(s). Can be a single file or a list
        that TwINFER merges when `merge_to_multiple_states=True`.

    sim_type : str
        Label describing the simulation condition (e.g., "A_to_B_low_kon").

    rep_id : int
        Replicate identifier for this simulation run.

    base_config : dict
        Configuration dictionary passed directly to TwINFER.

    t1, t2 : int or float
        Time points used for extracting twin measurements.

    gene_names : list[str], optional
        Names of genes used to label gene-pair matrix columns.
        If None, generic names are created.

    Returns
    -------
    record : dict
        A flattened dictionary containing:
        - simulation metadata
        - gene-gene correlation threshold for (gene_1, gene_2)
        - flattened directional matrix
        - flattened gene-gene correlation matrix
        - flattened random-pair correlation matrix (t2)
        - flattened twin-pair correlation matrices (t1 and t2)
    """

    results = infer_with_twinfer(
        path_to_simulation_file,
        merge_to_multiple_states=True,
        base_config=base_config,
        t1=t1, 
        t2=t2,
        check_for_steady_state=False,
        have_any_output=False, 
        show_scrambled_distribution_gene_correlation=False,
        plot_correlation_matrices_as_heatmap=False, 
        return_gene_corr_thresholds=True, 
        match_sim_details=False,
        seed=101010
    )

    # Base metadata
    record = {
        "sim_type": sim_type,
        "rep_id": rep_id,
        "analysis_key": f"{sim_type}_rep_{rep_id}",
        "gene_gene_threshold": results["gene_corr_thresholds"][("gene_1", "gene_2")]
    }

    def matrix_to_gene_pair_columns(matrix, gene_names, prefix=""):
        """
        Convert a square matrix (N×N) into flat dictionary columns labeled by
        'prefix + gene_i_gene_j'.

        Parameters
        ----------
        matrix : array-like or DataFrame
            Matrix to flatten.

        gene_names : list[str]
            Names for each row/column index.

        prefix : str
            Optional prefix (e.g., 'directional_', 'gene_gene_').

        Returns
        -------
        dict
            Flattened key → value mapping.
        """
        columns = {}
        if matrix is not None:
            matrix_array = np.array(matrix)
            n_genes = len(gene_names) if gene_names else matrix_array.shape[0]

            if gene_names is None:
                gene_names = [f"g{i+1}" for i in range(n_genes)]

            for i in range(matrix_array.shape[0]):
                for j in range(matrix_array.shape[1]):
                    key = f"{prefix}{gene_names[i]}_{gene_names[j]}"
                    columns[key] = matrix_array[i, j]
        return columns

    # All matrices we want to flatten
    matrix_data = {
        "directional": results["direction_matrix"],
        "gene_gene": results["pairwise_gene_gene_correlation_matrix"],
        "random": results["random_pair_correlation_matrix_t2"],
        "twin_t1": results["twin_pair_correlation_matrix_t1"],
        "twin_t2": results["twin_pair_correlation_matrix_t2"],
        "unfiltered": results['unfiltered_direction_matrix']
    }

    # Flatten matrices and add to output dictionary
    for mtype, matrix in matrix_data.items():
        record.update(matrix_to_gene_pair_columns(matrix, gene_names, f"{mtype}_"))
    return record


def save_results_to_csv(results_list, output_dir="results"):
    """
    Save flattened simulation results to multiple CSV files:
    - all_simulation_results.csv (everything combined)
    - one CSV per matrix type (directional, gene_gene, random, twin_t1, twin_t2)
    - summary CSV aggregated over simulation types

    Parameters
    ----------
    results_list : list[dict]
        List of flattened simulation records created by process_simulation().

    output_dir : str
        Directory where all files will be saved.

    Returns
    -------
    df : pandas.DataFrame
        Full combined DataFrame of all results.
    """
    df = pd.DataFrame(results_list)
    
    os.makedirs(output_dir, exist_ok=True)
    matrix_types = ["directional", "gene_gene", "random", "twin_t1", "twin_t2", "unfiltered"]

    # Create separate CSV for each matrix type
    for matrix_type in matrix_types:
        matrix_columns = [c for c in df.columns if c.startswith(f"{matrix_type}_")]

        if matrix_columns:
            metadata_cols = ["sim_type", "rep_id", "analysis_key"]
            subset_cols = metadata_cols + matrix_columns
            subset_df = df[subset_cols].copy()

            # Remove prefix for nicer output
            rename_dict = {col: col.replace(f"{matrix_type}_", "") for col in matrix_columns}
            subset_df.rename(columns=rename_dict, inplace=True)

            file_path = os.path.join(output_dir, f"{matrix_type}_matrix_results.csv")
            subset_df.to_csv(file_path, index=False)
            print(f"✅ Saved {matrix_type} matrix to {file_path}")
    return df

In [7]:
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import re

# Function to get all files matching a pattern
def get_all_rep_files(pattern):
    return glob.glob(pattern)

def process_all_reps_with_twins(file_pattern_or_folders, base_config, t1, t2, twin_data_df, sim_type_name, is_two_state=False):
    all_medians = []
    z_scores = []
    z_score_rand_list = []
    z_threshold_list = []
    if is_two_state:
        # Handle 2-state cases with paired files
        high_k_files = glob.glob(f"{file_pattern_or_folders[0]}/df_rows_*_rep_*.csv")
        low_k_files = glob.glob(f"{file_pattern_or_folders[1]}/df_rows_*_rep_*.csv")
        
        for high_file in high_k_files:
            # Extract rep number from filename
            rep_num = int(high_file.split('rep_')[-1].split('.csv')[0])
            
            # Find matching low_k file
            matching_low_file = [f for f in low_k_files if f'rep_{rep_num}.csv' in f]
            
            if matching_low_file:
                file_pair = [high_file, matching_low_file[0]]
                print(f"Processing {sim_type_name} rep {rep_num}: {file_pair}")
                
                try:
                    random_correlation = get_many_random_pair_corr(
                        file_pair,
                        base_config,
                        t1, t2,
                        check_for_steady_state=False,
                        have_any_output=False,
                        plot_correlation_matrices_as_heatmap=False
                    )["all_values"]
                    
                    # Calculate median and z-score
                    median_corr = np.median(random_correlation)
                    all_medians.append(median_corr)
                    
                    # Get twin data
                    twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) & 
                                           (twin_data_df['rep_id'] == rep_num)]
                    random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) & 
                                           (random_correlation_data['rep_id'] == rep_num)]
                    if not twin_row.empty:
                        twin_vals = twin_row['g2_g1'].values[0]
                        random_vals = random_row['g2_g1'].values[0]
                        random_mean = np.mean(random_correlation)
                        random_std = np.std(random_correlation)
                        z_score = (twin_vals - random_mean) / random_std
                        z_scores.append(z_score)
                        # z_threshold = 12*random_std + random_mean
                        z_threshold = -12*random_std + random_mean
                        z_threshold_list.append(z_threshold)
                        z_score_rand = (random_vals - random_mean) / random_std
                        z_score_rand_list.append(z_score_rand)
                    else:
                        print(f"Warning: No twin data found for {sim_type_name} rep {rep_num}")
                        z_scores.append(np.nan)
                        
                except Exception as e:
                    print(f"Error processing {file_pair}: {e}")
                    continue
    else:
        # Handle single-state cases
        files = glob.glob(file_pattern_or_folders)
        
        for file_path in files:
            rep_num = int(re.search(r'rep_(\d+)', file_path).group(1))
            print(f"Processing {sim_type_name} rep {rep_num}: {file_path}")
            
            try:
                random_correlation = get_many_random_pair_corr(
                    file_path,
                    base_config,
                    t1, t2,
                    check_for_steady_state=False,
                    have_any_output=False,
                    plot_correlation_matrices_as_heatmap=False
                )["all_values"]
                
                median_corr = np.median(random_correlation)
                all_medians.append(median_corr)
                
                # Get twin data
                twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) & 
                                    (twin_data_df['rep_id'] == rep_num)]
                random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) & 
                                        (random_correlation_data['rep_id'] == rep_num)]
                if not twin_row.empty:
                    twin_vals = twin_row['g2_g1'].values[0]
                    random_vals = random_row['g2_g1'].values[0]
                    random_mean = np.mean(random_correlation)
                    random_std = np.std(random_correlation)
                    z_score = (twin_vals - random_mean) / random_std
                    z_scores.append(z_score)
                    z_threshold = 10*random_std + random_mean
                    z_threshold_list.append(z_threshold)
                    z_score_rand = (random_vals - random_mean) / random_std
                    z_score_rand_list.append(z_score_rand)
                else:
                    print(f"Warning: No twin data found for {sim_type_name} rep {rep_num}")
                    z_scores.append(np.nan)
                    
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                continue
    
    return all_medians, z_scores, z_score_rand_list, z_threshold_list

## Analyze all simulations using TwINFER (run the pipeline using infer_with_twinfer function)

In [8]:
# #Collect all the simulations needed for figure 2: The 4 scenarios.
# tasks = []

# # Case 1: Fan out
# sim_folder = f"{path_to_simulation_data}"
# pattern = os.path.join(sim_folder, "df_rows_0_0_0_*_ncells_6000_Fan_out_*.csv")
# for i,f in enumerate(sorted(glob.glob(pattern))):
#     if "k_on" in f:
#         continue
#     if "additive" in f:
#         rep_id = os.path.basename(f)
#         tasks.append((f, "Fan_out", rep_id))

# # Case 2: Feed forward
# sim_folder = f"{path_to_simulation_data}"
# pattern = os.path.join(sim_folder, "df_rows_0_0_0_*_ncells_6000_Feed_forward_*.csv")
# for i,f in enumerate(sorted(glob.glob(pattern))):
#     if "k_on" in f:
#         continue
#     if "additive" in f:
#         rep_id = os.path.basename(f)
#         tasks.append((f, "Feed_forward", rep_id))

# # Case 3: Mutual regulation
# sim_folder = f"{path_to_simulation_data}"
# pattern = os.path.join(sim_folder, "df_rows_0_0_0_*_ncells_6000_Mutual_regulation_*.csv")
# for i,f in enumerate(sorted(glob.glob(pattern))):
#     if "k_on" in f:
#         continue
#     if "additive" in f:
#         rep_id = os.path.basename(f)
#         tasks.append((f, "Mutual_regulation", rep_id))

# Case 3: Fan in
sim_folder = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/saturation_effects/fan_in/"
pattern = os.path.join(sim_folder, "df_rows_0_0_0_*_ncells_6000_fan_in_k_add_6_*.csv")
for i,f in enumerate(sorted(glob.glob(pattern))):
    if "k_on" in f:
        continue
    if "additive" in f:
        rep_id = os.path.basename(f)
        tasks.append((f, "Fan_in", rep_id))


In [11]:
import os, glob
tasks = []
MAX_FILES_PER_MOTIF = 20

def add_tasks_first_n(tasks, sim_folder, pattern_tail, sim_type, max_n=20):
    pattern = os.path.join(sim_folder, pattern_tail)
    count = 0
    for f in glob.iglob(pattern):  # generator: does NOT build full list
        if "k_on" in f:
            continue
        rep_id = os.path.basename(f)
        tasks.append((f, sim_type, rep_id))
        count += 1
        if count >= max_n:
            break

# sim_folder = f"{path_to_simulation_data}"
sim_folder = "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/saturation_effects/fan_in/"

# add_tasks_first_n(tasks, sim_folder, "df_rows_0_0_0_*_ncells_6000_Fan_out_*.csv", "Fan_out", MAX_FILES_PER_MOTIF)
# add_tasks_first_n(tasks, sim_folder, "df_rows_0_0_0_*_ncells_6000_Feed_forward_*.csv", "Feed_forward", MAX_FILES_PER_MOTIF)
# add_tasks_first_n(tasks, sim_folder, "df_rows_0_0_0_*_ncells_6000_Mutual_regulation_*.csv", "Mutual_regulation", MAX_FILES_PER_MOTIF)
add_tasks_first_n(tasks, sim_folder, "df_rows_0_0_0_*_ncells_6000_fan_in_k_add_6_*.csv", "Fan_in", MAX_FILES_PER_MOTIF)


In [16]:
# # # # ---------- Run all tasks in parallel and save the results in plot_data folder
print("🚀 Starting parallel processing...")

results_list = Parallel(n_jobs=8, backend="loky")(
    delayed(process_simulation_through_infer_with_twinfer)(path, sim_type, rep_id, base_config, t1, t2)
    for path, sim_type, rep_id in tasks
)

# # ---------- Save results to CSV ----------
print("\n💾 Saving results to CSV files...")
df = save_results_to_csv(results_list, output_dir = path_to_save_plot_data)

# ---------- Quick analysis ----------
print(f"\n All simulations processed and saved!")
print(f"Total simulations: {len(df)}")
print(f"Simulation types: {list(df['sim_type'].unique())}")
print(f"Columns created: {len(df.columns)}")

🚀 Starting parallel processing...
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.

Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Only one simulation file was provided while merge_to_multiple_states was set to True. The file will be used as-is.
Could not ascertain corresponding parameter ro

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

path = f"{path_to_save_plot_data}/gene_gene_matrix_results.csv"
df = pd.read_csv(path)

plt.figure(figsize=(8, 4))

edge = "g1_g2"
sns.stripplot(
    data=df,
    x="sim_type",      # categorical
    y=edge,      # numeric metric (change if needed)
    jitter=0.25,
    size=4,
    alpha=0.6
)

plt.ylim(0,0.15)
plt.axhline(0.02, linestyle = "--")
plt.xlabel("Simulation type")
plt.ylabel("Spearman correlation")
plt.xticks(rotation=45, ha="right")
plt.title(edge)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

path = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4_23012026/directional_matrix_results.csv"
df = pd.read_csv(path)

plt.figure(figsize=(8, 4))

edge = "g1_g3"
sns.stripplot(
    data=df,
    x="sim_type",      # categorical
    y=edge,      # numeric metric (change if needed)
    jitter=0.25,
    size=4,
    alpha=0.6
)

plt.ylim(0,0.15)
plt.axhline(0.02, linestyle = "--")
plt.xlabel("Simulation type")
plt.ylabel("Spearman correlation")
plt.xticks(rotation=45, ha="right")
plt.title(edge)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

path = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4/twins_random_zscore_summary.csv"
df = pd.read_csv(path)
df_plot = df[(df['metric'] == "z_scores") & (df['gene_pair'] == "('gene_1', 'gene_2')")]
plt.figure(figsize=(8, 4))

sns.stripplot(
    data=df_plot,
    x="network_type",      # categorical
    y="values",      # numeric metric (change if needed)
    jitter=0.25,
    size=4,
    alpha=0.6
)

plt.axhline(-12, linestyle = "--")
plt.xlabel("Simulation type")
plt.ylabel("Z-score")
plt.xticks(rotation=45, ha="right")
plt.title("gene_1, gene_3")
plt.tight_layout()
plt.show()

In [ ]:
path_direction = "/home/gzu5140/Keerthana_b1042/grnInference/plot_data/figure_4/directional_matrix_results.csv"
df = pd.read_csv(path_direction)

plt.figure(figsize=(8, 4))

sns.stripplot(
    data=df,
    x="sim_type",      # categorical
    y="g1_g3",      # numeric metric (change if needed)
    jitter=0.25,
    size=4,
    alpha=0.6
)

plt.ylim(-0.01,0.2)
plt.xlabel("Simulation type")
plt.ylabel("Spearman correlation")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
# Function to get all files matching a pattern
def get_all_rep_files(pattern):
    return glob.glob(pattern)
def process_all_reps_with_twins(file_pattern_or_folders, base_config, t1, t2, twin_data_df, sim_type_name, is_two_state=False):
    all_medians = []
    z_scores = []
    z_score_rand_list = []
    z_threshold_list = []
    if is_two_state:
        # Handle 2-state cases with paired files
        high_k_files = glob.glob(f"{file_pattern_or_folders[0]}/df_rows_*_rep_*.csv")
        low_k_files = glob.glob(f"{file_pattern_or_folders[1]}/df_rows_*_rep_*.csv")
        for high_file in high_k_files:
            # Extract rep number from filename
            rep_num = int(high_file.split('rep_')[-1].split('.csv')[0])
            # Find matching low_k file
            matching_low_file = [f for f in low_k_files if f'rep_{rep_num}.csv' in f]
            if matching_low_file:
                file_pair = [high_file, matching_low_file[0]]
                print(f"Processing {sim_type_name} rep {rep_num}: {file_pair}")
                try:
                    random_correlation = get_many_random_pair_corr(
                        file_pair,
                        base_config,
                        t1, t2,
                        check_for_steady_state=False,
                        have_any_output=False,
                        plot_correlation_matrices_as_heatmap=False
                    )["all_values"]
                    # Calculate median and z-score
                    median_corr = np.median(random_correlation)
                    all_medians.append(median_corr)
                    # Get twin data
                    twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) &
                                           (twin_data_df['rep_id'] == rep_num)]
                    random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) &
                                           (random_correlation_data['rep_id'] == rep_num)]
                    if not twin_row.empty:
                        twin_vals = twin_row['g2_g1'].values[0]
                        random_vals = random_row['g2_g1'].values[0]
                        random_mean = np.mean(random_correlation)
                        random_std = np.std(random_correlation)
                        z_score = (twin_vals - random_mean) / random_std
                        z_scores.append(z_score)
                        z_threshold = 10*random_std + random_mean
                        z_threshold_list.append(z_threshold)
                        z_score_rand = (random_vals - random_mean) / random_std
                        z_score_rand_list.append(z_score_rand)
                    else:
                        print(f"Warning: No twin data found for {sim_type_name} rep {rep_num}")
                        z_scores.append(np.nan)
                except Exception as e:
                    print(f"Error processing {file_pair}: {e}")
                    continue
    else:
        gene_list = ["gene_1", "gene_2", "gene_3"]
        from itertools import combinations
        gene_pairs = list(combinations(gene_list, 2))
        # Handle single-state cases
        files = glob.glob(file_pattern_or_folders)
        files = [f for f in files if "k_on" not in f]
        pairwise_medians = {pair: [] for pair in gene_pairs}
        pairwise_z_scores = {pair: [] for pair in gene_pairs}
        pairwise_z_scores_rand = {pair: [] for pair in gene_pairs}
        pairwise_thresholds = {pair: [] for pair in gene_pairs}
        pairwise_thresholds_neg = {pair: [] for pair in gene_pairs}
        for i, file_path in enumerate(files):
            if "k_on" in file_path:
                continue
            rep_num = os.path.basename(file_path)
            print(f"Processing {sim_type_name} rep {rep_num}: {file_path}")
            twin_row = twin_data_df[(twin_data_df['sim_type'] == sim_type_name) &
                                   (twin_data_df['rep_id'] == rep_num)]
            random_row = random_correlation_data[(random_correlation_data['sim_type'] == sim_type_name) &
                                   (random_correlation_data['rep_id'] == rep_num)]
            if twin_row.empty:
                print(f"problem with {file_path} and {rep_num}")
                continue
            scrambled_random_corr = get_many_random_pair_corr(
                file_path,
                base_config,
                t1, t2,
                check_for_steady_state=False,
                have_any_output=False,
                plot_correlation_matrices_as_heatmap=False
            )
            for g1, g2 in gene_pairs:
                # column name in twin/random dfs
                colname = f"g{g1[-1]}_g{g2[-1]}"    # must match your naming: g2_g1
                # --- median of random values for this pair ---
                random_corr_values = scrambled_random_corr[(g1, g2)]["all_values"]
                median_corr = np.median(random_corr_values)
                pairwise_medians[(g1, g2)].append(median_corr)
                # --- twin & random data for this rep ---
                if not twin_row.empty:
                    twin_vals = twin_row[colname].values[0]
                    random_vals = random_row[colname].values[0]
                    random_mean = scrambled_random_corr[(g1, g2)]["global_mean"]
                    random_std  = scrambled_random_corr[(g1, g2)]["global_std"]
                    z_score = (twin_vals - random_mean) / random_std
                    pairwise_z_scores[(g1, g2)].append(z_score)
                    z_score_rand = (random_vals - random_mean) / random_std
                    pairwise_z_scores_rand[(g1, g2)].append(z_score_rand)
                    threshold = 12*random_std + random_mean
                    pairwise_thresholds[(g1, g2)].append(threshold)
                    threshold_neg = -12*random_std + random_mean
                    pairwise_thresholds_neg[(g1, g2)].append(threshold_neg)
                else:
                    # missing rep — behave exactly like your old code
                    pairwise_z_scores[(g1, g2)].append(np.nan)
    return (
    pairwise_medians,
    pairwise_z_scores,
    pairwise_z_scores_rand,
    pairwise_thresholds,
    pairwise_thresholds_neg
    )
# Usage:
# Single state
twin_correlation_t1_data = pd.read_csv(f"{path_to_save_plot_data}/twin_t1_matrix_results.csv")
random_correlation_data = pd.read_csv(f"{path_to_save_plot_data}/random_matrix_results.csv")
Mutual_regulation_medians, Mutual_regulation_z_scores, Mutual_regulation_z_rand_scores, Mutual_regulation_z_threshold_list, Mutual_regulation_z_threshold_list_neg = process_all_reps_with_twins(
    f"{path_to_simulation_data}/df_rows_0_0_0_*_ncells_6000_Mutual_regulation*.csv",
    base_config, t1, t2, twin_correlation_t1_data, "Mutual_regulation", is_two_state=False
)
Feed_forward_medians, Feed_forward_z_scores, Feed_forward_z_rand_scores, Feed_forward_z_threshold_list, Feed_forward_z_threshold_list_neg = process_all_reps_with_twins(
    f"{path_to_simulation_data}/df_rows_0_0_0_*_ncells_6000_Feed_forward*.csv",
    base_config, t1, t2, twin_correlation_t1_data, "Feed_forward", is_two_state=False
)
Fan_out_medians, Fan_out_z_scores, Fan_out_z_rand_scores, Fan_out_z_threshold_list, Fan_out_z_threshold_list_neg = process_all_reps_with_twins(
    f"{path_to_simulation_data}/df_rows_0_0_0_*_ncells_6000_Fan_out*.csv",
    base_config, t1, t2, twin_correlation_t1_data, "Fan_out", is_two_state=False
)
import pandas as pd
# Prepare a dictionary of all lists
data_dict = {
    "network_type": [],
    "metric": [],
    "values": [],
    "gene_pair": []


}
entries = [
    ("Mutual_regulation", "medians", Mutual_regulation_medians),
    ("Mutual_regulation", "z_scores", Mutual_regulation_z_scores),
    ("Mutual_regulation", "z_rand_scores", Mutual_regulation_z_rand_scores),
    ("Mutual_regulation", "z_threshold_list", Mutual_regulation_z_threshold_list),
    ("Mutual_regulation", "z_threshold_list_neg", Mutual_regulation_z_threshold_list_neg),
    ("Feed_forward", "medians", Feed_forward_medians),
    ("Feed_forward", "z_scores", Feed_forward_z_scores),
    ("Feed_forward", "z_rand_scores", Feed_forward_z_rand_scores),
    ("Feed_forward", "z_threshold_list", Feed_forward_z_threshold_list),
    ("Feed_forward", "z_threshold_list_neg", Feed_forward_z_threshold_list_neg),
    ("Fan_out", "medians", Fan_out_medians),
    ("Fan_out", "z_scores", Fan_out_z_scores),
    ("Fan_out", "z_rand_scores", Fan_out_z_rand_scores),
    ("Fan_out", "z_threshold_list", Fan_out_z_threshold_list),
    ("Fan_out", "z_threshold_list_neg", Fan_out_z_threshold_list_neg),
]
for network_type, metric, gene_pairwise_data in entries:
    for gene_pair_name in gene_pairwise_data.keys():
        values = gene_pairwise_data[gene_pair_name]
        for v in values:
            data_dict['gene_pair'].append(gene_pair_name)
            data_dict["network_type"].append(network_type)
            data_dict["metric"].append(metric)
            data_dict["values"].append(v)
df = pd.DataFrame(data_dict)
output_csv = f"{path_to_save_plot_data}/twins_random_zscore_summary_fin.csv"
df.to_csv(output_csv, index=False)
print("Saved to:", output_csv)

## Cross-correlation over time

In [ ]:
#!/usr/bin/env python3
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

def discover_motif_files(roots, rules, pattern="df*.csv", recursive=False, sort_by="mtime"):
    """
    Discover df*.csv files in roots, then assign them to motifs using ONLY filename checks.

    rules[motif]:
      - include_all: list[str] (filename must contain ALL substrings)  [AND]
      - exclude_any: list[str] (filename must contain NONE of these)
      - max_files: int|None    (keep newest N if sort_by='mtime')
    """
    candidates = []
    for root in roots:
        print(root)
        if recursive:
            candidates += glob.glob(os.path.join(root, "**", pattern), recursive=True)
        else:
            candidates += glob.glob(os.path.join(root, pattern))

    # de-dup and keep real files
    candidates = [p for p in dict.fromkeys(candidates) if p and os.path.isfile(p)]

    def _key(p):
        if sort_by == "name":
            return os.path.basename(p).lower()
        return os.path.getmtime(p)  # default: mtime

    out = {}
    for motif, r in rules.items():
        inc_all = [s.lower() for s in r.get("include_all", [])]
        exc_any = [s.lower() for s in r.get("exclude_any", [])]
        max_files = r.get("max_files", None)

        hits = []
        for p in candidates:
            name = os.path.basename(p).lower()

            # AND filter: must contain ALL required substrings
            if inc_all and not all(k in name for k in inc_all):
                continue

            # exclusions: must contain NONE of these
            if exc_any and any(k in name for k in exc_any):
                continue

            hits.append(p)

        if sort_by == "name":
            hits = sorted(hits, key=_key)
        else:
            hits = sorted(hits, key=_key, reverse=True)  # newest first

        if isinstance(max_files, int) and max_files > 0:
            hits = hits[:max_files]

        out[motif] = hits
        print(f"[discover] {motif}: {len(hits)} files")
        if hits:
            print("  example:", os.path.basename(hits[0]))

    return out


# ============================================================
# Helpers
# ============================================================
def _spearman(a, b):
    """Spearman rank correlation with pairwise finite masking."""
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return np.nan
    r, _ = spearmanr(a[mask], b[mask])
    return float(r)

def detect_twins_scheme(df: pd.DataFrame):
    """
    Decide which replicate labels are twin A and twin B for this file.

    - If replicates include 0 and 1  -> A = 0, B = 1
    - Else if replicates include 1 and 2 (and no 0) -> A = 1, B = 2
    - Otherwise, raise an error.
    """
    reps = set(df["replicate"].dropna().unique())
    if {0, 1}.issubset(reps):
        return 0, 1
    elif {1, 2}.issubset(reps) and 0 not in reps:
        return 1, 2
    else:
        raise ValueError(f"Cannot infer twin scheme from replicates: {sorted(reps)}")

# ============================================================
# Twins: cross-time Spearman (twin A @ t1, twin B @ t2)
# with HALF the clones used for cross-twin pairs
# ============================================================
def spearman_cross_twins_per_file_like_matrix(
    filepath,
    t1,
    t2,
    time_col=None,
    y_col=None,
    x1_col=None,
    x2_col=None,
    rep_t1=None,
    rep_t2=None,
    type_comparison="twin",  # "twin" or "random"
) -> pd.DataFrame:
    """
    Compute cross-time Spearman correlations using *half* the clones as real twins.

    For each undirected gene pair (g1, g2) in PAIR_ORDER, return:
      - Spearman(x_t1,y_t2): g1(t1, A) vs g2(t2, B)
      - Spearman(y_t1,x_t2): g2(t1, A) vs g1(t2, B)

    If type_comparison == "random", break twin pairing by permuting the t2 rows
    within the half-sampled set.
    """
    df = pd.read_csv(filepath)

    # decide repA/repB for THIS file
    if rep_t1 is None or rep_t2 is None:
        repA, repB = detect_twins_scheme(df)
    else:
        repA, repB = rep_t1, rep_t2

    base_cols = ["clone_id", "replicate", y_col, x1_col, x2_col]

    df_t1 = df.loc[(df[time_col] == t1) & (df["replicate"] == repA), base_cols].copy()
    df_t2 = df.loc[(df[time_col] == t2) & (df["replicate"] == repB), base_cols].copy()

    # average duplicates (if any)
    if df_t1.duplicated(subset=["clone_id"]).any():
        df_t1 = df_t1.groupby("clone_id", as_index=False).mean(numeric_only=True)
    if df_t2.duplicated(subset=["clone_id"]).any():
        df_t2 = df_t2.groupby("clone_id", as_index=False).mean(numeric_only=True)

    # strict twin safety check
    clones_t1 = set(df_t1["clone_id"])
    clones_t2 = set(df_t2["clone_id"])
    if type_comparison == "twin":
        if clones_t1 != clones_t2:
            only_t1 = sorted(clones_t1 - clones_t2)
            only_t2 = sorted(clones_t2 - clones_t1)
            print("[error] Twin mismatch between t1 and t2 for file:", filepath)
            print(f"  t1 (rep={repA}) clones: {len(clones_t1)}")
            print(f"  t2 (rep={repB}) clones: {len(clones_t2)}")
            if only_t1:
                print("  present only at t1 (first few):", only_t1[:10])
            if only_t2:
                print("  present only at t2 (first few):", only_t2[:10])
            raise ValueError("Mismatch in clone_id sets between t1 and t2 for twin comparison")

    rename_t1 = {col: f"{col}_t1" for col in [y_col, x1_col, x2_col]}
    rename_t2 = {col: f"{col}_t2" for col in [y_col, x1_col, x2_col]}
    df_t1 = df_t1.rename(columns=rename_t1)
    df_t2 = df_t2.rename(columns=rename_t2)

    merged = df_t1.merge(df_t2, on="clone_id", how="inner")
    if merged.empty:
        cols = ["gene_pair", "Spearman(x_t1,y_t2)", "Spearman(y_t1,x_t2)"]
        return pd.DataFrame(columns=cols).set_index("gene_pair")

    # HALF-sampling
    n_total = len(merged)
    if n_total >= 2:
        n_keep = n_total // 2
        idx = np.random.permutation(n_total)[:n_keep]
        merged = merged.iloc[idx].reset_index(drop=True)

    # random: break pairing
    if type_comparison == "random":
        n = len(merged)
        perm = np.random.permutation(n)
        merged_rand = merged.copy()
        for col in [f"{y_col}_t2", f"{x1_col}_t2", f"{x2_col}_t2"]:
            merged_rand[col] = merged[col].values[perm]
        used_df = merged_rand
    else:
        used_df = merged

    rows = []
    for (g1, g2) in PAIR_ORDER:
        x_forward = used_df[f"{g1}_t1"].values
        y_forward = used_df[f"{g2}_t2"].values
        r_forward = _spearman(x_forward, y_forward)

        x_reverse = used_df[f"{g2}_t1"].values
        y_reverse = used_df[f"{g1}_t2"].values
        r_reverse = _spearman(x_reverse, y_reverse)

        rows.append({
            "gene_pair": PAIR_LABELS[(g1, g2)],
            "Spearman(x_t1,y_t2)": r_forward,
            "Spearman(y_t1,x_t2)": r_reverse,
        })

    return pd.DataFrame(rows).set_index("gene_pair")

# ============================================================
# Twins: build tidy table over t2 for all files in a motif
# ============================================================
def build_tidy_for_files(files, motif_name, t1, t2_values):
    files = [p for p in dict.fromkeys(files) if p and os.path.isfile(p)]
    if not files:
        print(f"[warn] No valid files for motif: {motif_name}")
        return pd.DataFrame(columns=["motif","file","t1","t2","gene_pair","metric","value"])

    print(f"[info] {motif_name}: using {len(files)} twin files")
    records = []
    for fp in files:
        base = os.path.basename(fp)
        for t2 in t2_values:
            try:
                tbl = spearman_cross_twins_per_file_like_matrix(
                    fp, t1, t2,
                    time_col=time_col,
                    y_col=y_col,
                    x1_col=x1_col,
                    x2_col=x2_col,
                    type_comparison="twin",
                )
            except Exception as e:
                print(f"[warn] {motif_name} | {base} | t2={t2}: {e}")
                continue

            for gp, row in tbl.iterrows():
                for metric in METRICS_TO_PLOT:
                    val = row.get(metric, np.nan)
                    records.append({
                        "motif":     motif_name,
                        "file":      base,
                        "t1":        int(t1),
                        "t2":        int(t2),
                        "gene_pair": gp,
                        "metric":    metric,
                        "value":     None if pd.isna(val) else float(val),
                    })

    return pd.DataFrame.from_records(records)

def build_tidy_regular_for_files(files, motif_name, t_values=None, t1=None):
    files = [p for p in dict.fromkeys(files) if p and os.path.isfile(p)]
    if not files:
        print(f"[warn] No valid files for motif (regular): {motif_name}")
        return pd.DataFrame(columns=["motif","file","t1","t2","gene_pair","metric","value"])

    print(f"[info-pop] {motif_name}: computing population corr at each time")
    records = []

    for fp in files:
        base = os.path.basename(fp)
        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print(f"[warn-pop] {motif_name} | {base}: {e}")
            continue

        try:
            repA, repB = detect_twins_scheme(df)
        except ValueError as e:
            print(f"[warn-pop] {motif_name} | {base}: {e}")
            continue

        if t_values is None:
            times = np.sort(df[time_col].dropna().unique())
        else:
            times = np.array(sorted(set(int(t) for t in t_values)))

        for t in times:
            sub = df[(df["replicate"] == repB) & (df[time_col] == t)].dropna(
                subset=[x1_col, x2_col, y_col]
            )
            if sub.empty:
                continue

            rho_xy = _spearman(sub[x1_col], sub[x2_col])
            rho_zy = _spearman(sub[y_col],  sub[x2_col])
            rho_zx = _spearman(sub[y_col],  sub[x1_col])

            for (g1, g2) in PAIR_ORDER:
                gp = PAIR_LABELS[(g1, g2)]
                if gp == "X-Y":
                    rho = rho_xy
                elif gp == "Z-Y":
                    rho = rho_zy
                elif gp == "Z-X":
                    rho = rho_zx
                else:
                    continue

                records.append({
                    "motif":     motif_name,
                    "file":      base,
                    "t1":        int(t),
                    "t2":        int(t),
                    "gene_pair": gp,
                    "metric":    "Spearman_same_time",
                    "value":     float(rho),
                })

    return pd.DataFrame.from_records(records)


# ============================================================
# Main
# ============================================================
t1_fixed = 1
t2_values = list(range(1, 49))   # 1..48
time_col = 'time_step'

# Gene columns (Z, X, Y)
y_col  = 'gene_1_mRNA'  # Z
x1_col = 'gene_2_mRNA'  # X
x2_col = 'gene_3_mRNA'  # Y

# Metrics to compute / plot
METRICS_TO_PLOT = ["Spearman(x_t1,y_t2)", "Spearman(y_t1,x_t2)"]

# Label the genes as Z, X, Y (for legend text)
GENE_LABELS = {y_col: "Z", x1_col: "X", x2_col: "Y"}

# Pair order (undirected pairs) and labels:
#   X-Y, Z-Y, Z-X
PAIR_ORDER  = [
    (x1_col, x2_col),   # X-Y
    (y_col,  x2_col),   # Z-Y
    (y_col,  x1_col),   # Z-X
]
PAIR_LABELS = {(a, b): f"{GENE_LABELS[a]}-{GENE_LABELS[b]}" for (a, b) in PAIR_ORDER}


# ============================================================
# AUTO-DISCOVERY: folders + filename keywords only
# include_all = AND logic (must contain ALL keywords)
# ============================================================

MOTIF_RULES = {
    "Fan-out": {
        "include_all": ["Fan_out", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,   # set int (e.g., 20) to cap newest files
    },
    "Feed-forward Loop": {
        "include_all": ["Feed_forward", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,
    },
    "Regulated Mutual": {
        "include_all": ["Mutual_regulation", "df_rows"],
        "exclude_any": ["k_on0"],
        "max_files": 20,
    },
}

MOTIF_FILES = discover_motif_files(
    [path_to_simulation_data],
    MOTIF_RULES,
    pattern="df*.csv",
    recursive=False,   # set True if your files are in subfolders
    sort_by="mtime",
)


In [ ]:
all_tidy_twin = []
for motif, files in MOTIF_FILES.items():
    tidy_motif = build_tidy_for_files(files, motif, t1_fixed, t2_values)
    if not tidy_motif.empty:
        all_tidy_twin.append(tidy_motif)


        
if not all_tidy_twin:
    raise SystemExit("[abort] No twin data collected. Check path_to_simulations / MOTIF_RULES.")
tidy_twin = pd.concat(all_tidy_twin, ignore_index=True)
all_tidy_reg = []
for motif, files in MOTIF_FILES.items():
    tidy_reg_motif = build_tidy_regular_for_files(files, motif, t_values=t2_values, t1=t1_fixed)
    if not tidy_reg_motif.empty:
        all_tidy_reg.append(tidy_reg_motif)
if not all_tidy_reg:
    raise SystemExit("[abort] No regular-corr data collected. Check path_to_simulations / MOTIF_RULES.")
tidy_reg = pd.concat(all_tidy_reg, ignore_index=True)

In [28]:
tidy_reg.to_csv(f"{path_to_save_plot_data}/gene_correlation_all.csv")

In [29]:
tidy_twin.to_csv(f"{path_to_save_plot_data}/twin_correlation_all.csv")

### Figure 4c

In [5]:
# Calculation functions
import sys
sys.path.append(str(path_to_code_repo))
import importlib
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers
from TwINFER_function_scripts import infer_with_twinfer

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)
importlib.reload(infer_with_twinfer)

from TwINFER_function_scripts.correlation_analysis_functions import (
    generate_random_shuffle
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    split_and_merge_simulations,
    get_param_data, 
    plot_matrix_as_heatmap,
)

from TwINFER_function_scripts.infer_with_twinfer import (
    infer_with_twinfer
)

In [70]:
protein_1_2 = 125265.1379158951
protein_1 = 29965.104
protein_2_ffl = 119763.6917
protein_2_cascade = 119763.6917
protein_3_cascade = 122672.08966666667
mutual_reg_2_3 = 125265.1379158951
K_matrix = [
    [0, protein_1, protein_1, 0, 0,0,0,0,0,0,0,0,0,0],
    [0, 0, 0, protein_2_ffl, 0,0,0,0,0,0,0,0,0,0],
    [0]*14,
    [0]*14,
    [0, 0, 0, 0, 0, protein_1, protein_1, 0, 0,0,0,0,0,0],
    [0,0,0,0,0,0,protein_2_ffl,protein_2_ffl,0,0,0,0,0,0],
    [0]*14,
    [0]*14,
    [0]*14,
    [0,0,0,0,0,0,0,0,0,0,protein_1,0,0,0],
    [0,0,0,0,0,0,0,0,0,0, 0, protein_2_cascade, 0, 0],
    [0,0,0,0,0,0,0,0,0,0, 0,0, protein_3_cascade,protein_3_cascade],
    [0,0,0,0,0,0,0,0,0,0,0,0,0,mutual_reg_2_3],
    [0,0,0,0,0,0,0,0,0,0,0,0,mutual_reg_2_3,0]
]

base_config_figure_1 = {
    'n_cells': 6000,
    'simulation_time_before_division': 6000,
    'twin_simulation_time_after_division': 48,
    'rows_to_use':[[0]*14],
    'twin_measurement_resolution': 1,
    "path_to_connectivity_matrix": f"{path_to_code_repo}/simulation_example_input_data/connectivity_matrix_figure_1_network.txt",  # Will be updated per network type
    "param_csv":f"{path_to_code_repo}/simulation_example_input_data/median_parameter.csv",  # Will be updated per network type
    "output_folder": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/figure_1_network",
    "log_file": "/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/figure_1_network/figure_1_network.jsonl",
    "type": "figure_1_network",
    "use_given_K": True,
    "K_to_use":K_matrix,
    "multiple_interaction_type": "additive",
    "number_of_parallel_parameters": 1,
    "number_of_cores_per_parameter": 56,
    "log_pi_on": False,
}

path_to_simulation_file = f"/home/gzu5140/Keerthana_b1042/grnInference/simulation_data/figure_4/df_rows_0_0_0_0_0_0_0_0_0_0_0_0_0_0_19012026_072143_ncells_6000_figure_1_network_check_0_2_393d9d82.csv"

In [90]:
twinfer_kwargs_figure_1 = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config_figure_1,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.002, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": False, 
    "z_score_threshold_two_states": 12, #Z-score False to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.01,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": False,
    "seed": 101010,
    "infer_direction_for_which_edges":"all-potential-regulation", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 12
}

In [ ]:
# Process each network type
all_correlation_matrices = {}
   
# Use the first CSV file found (or you can add logic to select specific one)
print(f"Using simulation file: {os.path.basename(path_to_simulation_file)}")
    
# Update config for this network type
config = base_config_figure_1
network_type = "figure_1_network" 
config["type"] = {network_type}

def make_json_safe(obj):
    if hasattr(obj, "to_dict"):      # pandas DataFrame / Series
        return obj.to_dict()
    if isinstance(obj, set):
        return list(obj)
    return obj
    
# Check if required files exist
if not os.path.exists(config["path_to_connectivity_matrix"]):
    print(f"Warning: Connectivity matrix not found for {network_type}")

if not os.path.exists(config["param_csv"]):
    print(f"Warning: Parameter CSV not found for {network_type}")
    
try:
    # Run inference for this network type
    correlation_matrices = infer_with_twinfer(
           **twinfer_kwargs_figure_1
        )
        

except Exception as e:
    print(f"Error processing {network_type}: {str(e)}")

In [ ]:
import json
# Store the correlation matrices
all_correlation_matrices[network_type] = correlation_matrices
    
# Save the directional correlation matrix for this network type
json_safe = {
    k: make_json_safe(v)
    for k, v in correlation_matrices.items()
}
path_to_json_file = f"{path_to_save_plot_data}figure_1_network_correlation_matrices.json"
with open(path_to_json_file, "w") as f:
    json.dump(json_safe, f, indent=2)
print(correlation_matrices.keys())
correlation_file_name = f"{path_to_save_plot_data}/filtered_directional_correlation_type_{network_type}.csv"
gene_correlation_file_name = f"{path_to_save_plot_data}/gene_correlation_type_{network_type}.csv"
correlation_matrices['unfiltered_direction_matrix'].to_csv(correlation_file_name)
correlation_matrices['pairwise_gene_gene_correlation_matrix'].to_csv(gene_correlation_file_name)
correlation_matrices
print(f"Successfully processed {network_type}")
print(f"Saved correlation matrix to: {correlation_file_name}")

In [ ]:
correlation_matrices.keys()